In [ ]:
import numpy as np
import pandas as pd

In [ ]:
interpolate_cols = {
    "passengers_acceptance_rate",
    "drivers_pickup",
    "drivers_idle",
    "drivers_busy",
    "drivers_demand_pressure",
    "drivers_acceptance_rate",
    "rides_offers_radius_avg",
    "rides_offers_price_avg",
    "rides_offers_price_lyft_avg",
    "rides_offers_price_uber_avg",
    "rides_offers_surge_avg",
    "rides_offers_surge_lyft_avg",
    "rides_offers_surge_uber_avg",
    "rides_offers_price_no_surge_avg",
    "rides_duration_expected_avg",
    "rides_length_expected_avg",
    "passengers_drivers_ratio",
    "passengers_rating_avg"
}
deaggregate_cols = {
    "passengers_unassigned",
    "passengers_assigned",
    "passengers_accept",
    "passengers_reject",
    "passengers_cancel",
    "passengers_offers_discarded",
    "drivers_accept",
    "drivers_reject",
    "drivers_removed",
    "rides_offers_generated",
    "rides_offers_uber",
    "rides_offers_lyft",
    "rides_dispatched",
    "rides_partial_acceptances",
    "rides_not_served",
    "drivers_starting",
    "drivers_waiting",
    "drivers_change_zone",
    "drivers_change_provider",
    "drivers_pickup_accept",
    "drivers_pickup_reject",
    "passengers_change_provider",
    "passengers_pickup_accept",
    "passengers_pickup_reject",
    "passengers_no_ratings_count",
    "passengers_ratings_count"
}

In [ ]:
def fill_missing_data(
        df: pd.DataFrame,
        window: int = 60
    ) -> pd.DataFrame:
    """
    For columns that are zero at all non-multiples of 'window':
      - If the column is a snapshot -> interpolate between update points.
      - Else (assumed window-sum)   -> de-aggregate evenly over the window.

    Parameters
    ----------
    - df: pd.DataFrame
        Input dataframe with a 'timestamp' column and other feature columns.
    - window: int
        The interval at which certain columns are updated (default is 60).

    Returns
    -------
    pd.DataFrame
        DataFrame with missing data filled in according to the specified rules.
    """
    df = df.copy()
    t = df["timestamp"]
    update_mask = (t % window == 0)
    n = len(df)

    for col in df.columns[1:]:
        # Check if this column only updates every 'window' ticks
        nonzero_between = (df[col].ne(0) & ~update_mask).any()
        if nonzero_between:
            continue
        # Keep values only at multiples of 'window'
        c = df[col].where(update_mask)
        if col in interpolate_cols:
            # Interpolate linearly along integer index
            df[col] = c.astype(float).interpolate(method="linear", limit_direction="both").astype(float)
        elif col in deaggregate_cols:
            # De-aggregate uniformly across the window
            per_tick = np.zeros(n, dtype=float)
            upd_idx = np.flatnonzero(update_mask & c.notna())
            for end in upd_idx:
                # Split 'total' into 'length' integer parts whose sum equals 'total'
                start = max(0, end - (window - 1))
                length = end - start + 1
                val = int(round(c.iat[end]))
                q, r = divmod(val, length)
                parts = np.full(length, q, dtype=int)
                if r:
                    parts[-r:] += 1
                per_tick[start:end + 1] = parts
            df[col] = per_tick
    return df

In [ ]:
def scale_failed_requests(df: pd.DataFrame) -> tuple:
    """
    Scale the failed requests based on the number of passengers.

    Parameters
    ----------
    df: pd.DataFrame
        Input dataframe containing the 'passengers_new' column.

    Returns
    -------
    tuple
        Coefficients (a, b, c) for the scaling function.
    """
    # Define the values for x1, x2 and their corresponding y values
    x1, y1 = 0, 0
    x2, y2 = df['passengers_new'].mean(), 1 # Scaling factor = 1 when rides requested >= mean
    # Adjust b to ensure the function is defined at x = 0
    b = 0.1
    # Values of ln(x+b) at x1 and x2
    ln_x1_b = np.log(x1 + b)
    ln_x2_b = np.log(x2 + b)
    # System of equations:
    # a * ln(0.1) + c = 0
    # a * ln(df['rides_requested'].mean()) + c = 1
    # We can represent this system as matrix equations A * [a, c] = Y
    A = np.array([
        [ln_x1_b, 1],
        [ln_x2_b, 1]
    ])
    Y = np.array([y1, y2])
    # Solve for coefficients a and c
    coefficients = np.linalg.solve(A, Y)
    a, c = coefficients
    return a, b, c

In [ ]:
def compute_failed_requests(
        a: float, 
        b: float, 
        c: float, 
        failed: int, 
        requested: int
    ) -> float:
    """
    Compute the weighted failed requests based on the scaling function.

    Parameters
    ----------
    - a: float
        Coefficient for the logarithmic term.
    - b: float
        Offset for the logarithmic term.
    - c: float
        Constant term.
    - failed: int
        Number of failed requests.
    - requested: int
        Number of requested rides.

    Returns
    -------
    float
        Weighted failed requests.
    """
    if requested == 0:
        return 0
    failed_perc = failed / requested
    weight = min(1, a*np.log(requested+b)+c)
    w_failed_perc = failed_perc * weight
    return w_failed_perc

In [ ]:
def compute_time_above(
        hardiness: np.ndarray,
        t: int, 
        threshold_h: float
    ) -> list:
    """
    Compute the maximum consecutive timestamps where the hardiness is above a threshold.

    Parameters
    ----------
    - hardiness: np.ndarray
        Array of hardiness values.
    - t: int
        Time window for comparison.
    - threshold_h: float
        Hardiness threshold.

    Returns
    -------
    list
        List of maximum consecutive timestamps above the threshold.
    """
    timestamps = 0
    max_timestamps = []
    for i in range(t, len(hardiness)):
        if np.mean(hardiness[i-60:i]) > np.mean(hardiness[i-t:i-60]) or hardiness[i] > threshold_h:
            timestamps += 1
        else:
            if timestamps != 0:
                max_timestamps.append(timestamps)
            timestamps = 0
    return max_timestamps

In [ ]:
def exceed_threshold(
        index: np.ndarray,
        threshold: float
    ) -> list:
    """
    Find the indices where the values in the array exceed the given threshold.

    Parameters
    ----------
    - index: np.ndarray
        Array of values to check.
    - threshold: float
        Threshold value.

    Returns
    -------
    list
        List of indices where the values exceed the threshold.
    """
    idx = np.where(index > threshold)[0]
    return idx.tolist() if idx.size > 0 else []

In [ ]:
def get_first_above_threshold(
        exceed_list: list, 
        gap_tolerance: int
    ) -> list:
    """
    Get the first indices above the threshold considering a gap tolerance.

    Parameters
    ----------
    - exceed_list: list
        List of indices where the threshold is exceeded.
    - gap_tolerance: int
        Maximum allowed gap between consecutive indices.

    Returns
    -------
    list
        List of first indices above the threshold considering the gap tolerance.
    """
    if exceed_list:
        exceed_arr = np.array(exceed_list)
        mask = np.diff(exceed_arr) > gap_tolerance
        return exceed_arr[np.insert(mask, 0, True)].tolist()
    return []

In [ ]:
def compute_resilience(
        hardiness: np.ndarray,
        threshold_r: int,
        t: int,
        threshold_h: float
    ) -> tuple:
    """
    Find every occasion in which the hardiness stays above the trend/threshold
    for more than 'threshold_r' consecutive timestamps.

    Parameters
    ----------
    - hardiness: np.ndarray
        Array of hardiness values.
    - threshold_r: int
        Minimum number of consecutive timestamps above the threshold to consider.
    - t: int
        Time window for comparison.
    - threshold_h: float
        Hardiness threshold.

    Returns
    -------
    - exceeding_indices: list
        Timestamp at which each exceeding run crosses 'threshold_r'.
    - consecutive_timestamps: list
        Total length of each of those exceeding runs (same order/length as above).
    - max_timestamps: int
        Longest run observed, exceeding or not.
    """
    timestamps = 0
    max_timestamps = 0
    exceeding_indices = []
    consecutive_timestamps = []
    found = False
    for i in range(7200, len(hardiness)):
        if np.mean(hardiness[i-60:i]) > np.mean(hardiness[i-t:i-60]) or hardiness[i] > threshold_h:
            timestamps += 1
            if timestamps > threshold_r and not found:
                exceeding_indices.append(i)
                found = True
        else:
            if found:
                consecutive_timestamps.append(timestamps)
            max_timestamps = max(timestamps, max_timestamps)
            timestamps = 0
            found = False
    # Close the run still open at the end of the series
    if found:
        consecutive_timestamps.append(timestamps)
    max_timestamps = max(timestamps, max_timestamps)
    return exceeding_indices, consecutive_timestamps, max_timestamps

In [ ]:
def compute_thresholds(
        df: pd.DataFrame,
        aggregate: int,
        cut: int
    ) -> tuple:
    """
    Compute thresholds for hardiness, consistency, and resilience based on the given DataFrame.

    Parameters
    ----------
    - df: pd.DataFrame
        Input DataFrame containing the relevant columns.
    - aggregate: int
        Window size for rolling aggregation.
    - cut: int
        Number of initial and final rows to cut after aggregation.

    Returns
    -------
    - perc_99_h: float
        99th percentile threshold for hardiness.
    - perc_99_c: float
        99th percentile threshold for consistency.
    - perc_99_r: int
        99th percentile threshold for resilience.
    - a: float
        Scaled parameter a for computing failed requests.
    - b: float
        Scaled parameter b for computing failed requests.
    - c: float
        Scaled parameter c for computing failed requests.
    """
    # Aggregate data using average
    for column in df.columns:
        df[column] = df[column].rolling(window=aggregate).mean()
        df[column] = np.around(df[column].astype(float), 5)
    df = df[3600 + cut - aggregate:-cut].reset_index(drop=True)
    a, b, c = scale_failed_requests(df)
    # Hardiness
    failed_requests = [0] * len(df)
    for i in range(0, len(df)):
        failed_requests[i] = compute_failed_requests(
            a, b, c,
            failed=df['rides_not_served'][i] + df['passengers_cancel'][i],
            requested=df['passengers_new'][i]
        )
    hardiness = pd.Series(failed_requests)
    perc_99_h = float(np.percentile(hardiness, 99))
    # Consistency
    consistency = hardiness.rolling(window=aggregate).std().fillna(0).reset_index(drop=True)
    perc_99_c = float(np.percentile(consistency, 99))
    # Resilience
    consecutive_r_list = compute_time_above(hardiness, 1800, perc_99_h)
    perc_99_r = int(np.percentile(consecutive_r_list, 99))
    return perc_99_h, perc_99_c, perc_99_r, a, b, c

In [ ]:
def compute_exceeding_timestamps(
        df: pd.DataFrame,
        aggregate: int,
        cut: int,
        a: float,
        b: float,
        c: float,
        perc_99_h: float,
        perc_99_c: float,
        perc_99_r: int
    ) -> tuple:
    """
    Compute the timestamps at which hardiness, consistency, and resilience exceed their respective thresholds.

    Parameters
    ----------
    - df: pd.DataFrame
        Input DataFrame containing the relevant columns.
    - aggregate: int
        Window size for rolling aggregation.
    - cut: int
        Number of initial and final rows to cut after aggregation.
    - a: float
        Scaled parameter a for computing failed requests.
    - b: float
        Scaled parameter b for computing failed requests.
    - c: float
        Scaled parameter c for computing failed requests.
    - perc_99_h: float
        99th percentile threshold for hardiness.
    - perc_99_c: float
        99th percentile threshold for consistency.
    - perc_99_r: int
        99th percentile threshold for resilience.

    Returns
    -------
    - hardiness_event: pd.Series
        Series of hardiness values.
    - exceed_timestamp_h: list
        Timestamps at which hardiness exceeds the threshold.
    - exceed_timestamp_c: list
        Timestamps at which consistency exceeds the threshold.
    - exceed_timestamp_r: list
        Timestamps at which resilience exceeds the threshold.
    - consecutive_t: list
        Total length of each of the exceeding runs for resilience.
    """
    df.drop(columns=["timestamp"], inplace=True)
    # Aggregate data using average
    for column in df.columns:
        df[column] = df[column].rolling(window=aggregate).mean()
        df[column] = np.around(df[column].astype(float), 5)
    df = df[3600 + cut - aggregate:-cut].reset_index(drop=True)
    # Hardiness
    failed_requests_event = [0] * len(df)
    for i in range(0, len(df)):
        failed_requests_event[i] = compute_failed_requests(
            a, b, c,
            failed=df['rides_not_served'][i] + df['passengers_cancel'][i],
            requested=df['passengers_new'][i]
        )
    hardiness_event = pd.Series(failed_requests_event)
    exceed_timestamp_h = exceed_threshold(hardiness_event, perc_99_h)
    # Consistency
    consistency_event = hardiness_event.rolling(window=aggregate).std().fillna(0).reset_index(drop=True)
    exceed_timestamp_c = exceed_threshold(consistency_event, perc_99_c)
    # Resilience (one entry per exceeding occasion)
    exceed_timestamp_r, consecutive_t, _ = compute_resilience(hardiness_event, perc_99_r, 1800, perc_99_h)
    return hardiness_event, exceed_timestamp_h, exceed_timestamp_c, exceed_timestamp_r, consecutive_t

In [ ]:
def find_hcs_failure(
        exceed_timestamp_h: list,
        exceed_timestamp_c: list,
        exceed_timestamp_r: list,
        consecutive_t: list,
        perc_99_r: int
    ) -> int:
    """
    Return the first point at which the three indexes are out of range together. 
    Returns 0 when the run does not fail.

    Parameters
    ----------
    - exceed_timestamp_h: list
        Timestamps at which the hardiness index exceeds its threshold.
    - exceed_timestamp_c: list
        Timestamps at which the consistency index exceeds its threshold.
    - exceed_timestamp_r: list
        Timestamps at which the resilience index exceeds its threshold.
    - consecutive_t: list
        Lengths of consecutive runs for the resilience index.
    - perc_99_r: int
        The 99th percentile of the resilience run lengths.

    Returns
    -------
    int
        The first timestamp at which the HCS fails, or 0 if it does not fail.
    """
    for exceed_r, run_length in zip(exceed_timestamp_r, consecutive_t):
        # The index is recorded once the run has lasted perc_99_r + 1 timestamps,
        # so the run itself starts perc_99_r timestamps earlier.
        window_start = exceed_r - perc_99_r
        window_end = window_start + run_length
        h_in_window = [t for t in exceed_timestamp_h if window_start <= t <= window_end]
        c_in_window = [t for t in exceed_timestamp_c if window_start <= t <= window_end]
        if len(h_in_window) > 0 and len(c_in_window) > 0:
            return max(h_in_window[0], c_in_window[0], exceed_r)
    return 0